In [1]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

In [2]:
emi_df = pd.read_csv(r'D:\Emi Predict\Data Sets\emi_prediction_cleanafterFE.csv')

features = [
    "monthly_salary","school_fees","college_fees","travel_expenses",
    "groceries_utilities","other_monthly_expenses",
    "current_emi_amount","credit_score","requested_amount","bank_balance","emergency_fund"
]

X = emi_df[features]

In [3]:
y_clf = emi_df['emi_eligibility']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42
)

XGB Boost classification

In [4]:
y_clf = y_clf.apply(lambda x: 1 if x >= 1 else 0).astype(int)

clf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss"
    ))
])

clf_pipeline.fit(X_train, y_train)

print("Classification model saved")

Classification model saved


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Predictions
y_pred = clf_pipeline.predict(X_test)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred,average='weighted')
recall = recall_score(y_test, y_pred,average='weighted')
f1 = f1_score(y_test, y_pred,average='weighted')

print("📊 Classification Metrics")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Full Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

📊 Classification Metrics
Accuracy  : 0.8552
Precision : 0.8148
Recall    : 0.8552
F1 Score  : 0.8341

Confusion Matrix:
[[50627     0  2724]
 [ 2171     0   811]
 [ 4277     0  8321]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.95      0.92     53351
           1       0.00      0.00      0.00      2982
           2       0.70      0.66      0.68     12598

    accuracy                           0.86     68931
   macro avg       0.53      0.54      0.53     68931
weighted avg       0.81      0.86      0.83     68931



d:\Emi Predict\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Emi Predict\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Emi Predict\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Emi Predict\.venv\Lib\site-packages\s

In [21]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import xgboost as xgb

# Target
y_clf = emi_df['emi_eligibility']
y_clf = y_clf.apply(lambda x: 1 if x >= 1 else 0).astype(int)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42
)

# Pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", xgb.XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False
    ))
])

# 🔥 Hyperparameter space
param_dist = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__gamma": [0, 0.1, 0.3, 0.5],
    "model__min_child_weight": [1, 3, 5, 7]
}

# Random Search
random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=70,              # increase to 50+ for better results
    scoring="f1",           # better than accuracy
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# Train
random_search.fit(X_train, y_train)

# Best model
xgb_model = random_search.best_estimator_

# Save
pickle.dump(xgb_model, open(r"D:\Emi Predict\Models-1\clas_xgb.pkl", "wb"))

print("✅ Best model saved")

# Results
print("\nBest Parameters:")
print(random_search.best_params_)

# Evaluate
y_pre_xgb = xgb_model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pre_xgb))

accuracy = accuracy_score(y_test, y_pre_xgb)
precision = precision_score(y_test, y_pre_xgb)
recall = recall_score(y_test, y_pre_xgb)
f1 = f1_score(y_test, y_pre_xgb)

print("📊 Classification Metrics")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pre_xgb))

# Full Report
print("\nClassification Report:")
print(classification_report(y_test, y_pre_xgb))

Fitting 3 folds for each of 70 candidates, totalling 210 fits


d:\Emi Predict\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:12:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Best model saved

Best Parameters:
{'model__subsample': 0.8, 'model__n_estimators': 300, 'model__min_child_weight': 5, 'model__max_depth': 8, 'model__learning_rate': 0.05, 'model__gamma': 0.3, 'model__colsample_bytree': 0.9}

Accuracy: 0.8729889309599455
📊 Classification Metrics
Accuracy  : 0.8730
Precision : 0.7323
Recall    : 0.6904
F1 Score  : 0.7108

Confusion Matrix:
[[49419  3932]
 [ 4823 10757]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.93      0.92     53351
           1       0.73      0.69      0.71     15580

    accuracy                           0.87     68931
   macro avg       0.82      0.81      0.81     68931
weighted avg       0.87      0.87      0.87     68931



LOGISTIC REGRESSION

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pickle
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("📊 Classification Metrics")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Full Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

📊 Classification Metrics
Accuracy  : 0.8579
Precision : 0.7228
Recall    : 0.6022
F1 Score  : 0.6570

Confusion Matrix:
[[49752  3599]
 [ 6197  9383]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91     53351
           1       0.72      0.60      0.66     15580

    accuracy                           0.86     68931
   macro avg       0.81      0.77      0.78     68931
weighted avg       0.85      0.86      0.85     68931



In [25]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import pickle

# Create pipeline (scaling + model together)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=5000))
])

param_grid = [
    {
        'logreg__penalty': ['l2'],
        'logreg__C': [0.01, 0.1, 1, 10, 100],
        'logreg__solver': ['lbfgs', 'saga']
    },
    {
        'logreg__penalty': ['l1'],
        'logreg__C': [0.01, 0.1, 1, 10],
        'logreg__solver': ['liblinear', 'saga']
    },
    {
        'logreg__penalty': ['elasticnet'],
        'logreg__C': [0.01, 0.1, 1, 10],
        'logreg__solver': ['saga'],
        'logreg__l1_ratio': [0.1, 0.5, 0.9]
    }
]

# Grid Search
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',  # or 'roc_auc' if classification is imbalanced
    n_jobs=-1,
    verbose=1
)

# Fit
grid_search.fit(X_train, y_train)

# Best model
log_model = grid_search.best_estimator_

# Predictions
y_pred = log_model.predict(X_test)

# Save model
pickle.dump(log_model, open(r"D:\Emi Predict\Models-1\log.pkl", "wb"))

# Print best params
print("Best Parameters:", grid_search.best_params_)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("📊 Classification Metrics")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Full Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 30 candidates, totalling 150 fits


d:\Emi Predict\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best Parameters: {'logreg__C': 0.01, 'logreg__penalty': 'l2', 'logreg__solver': 'lbfgs'}
📊 Classification Metrics
Accuracy  : 0.8579
Precision : 0.7235
Recall    : 0.6010
F1 Score  : 0.6566

Confusion Matrix:
[[49773  3578]
 [ 6216  9364]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91     53351
           1       0.72      0.60      0.66     15580

    accuracy                           0.86     68931
   macro avg       0.81      0.77      0.78     68931
weighted avg       0.85      0.86      0.85     68931



In [22]:
import pickle
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Pipeline (scaling + polynomial features + model)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('logreg', LogisticRegression(max_iter=5000))
])

# Parameter distribution (wider + smarter search)
param_dist = {
    'poly__degree': [1, 2],
    'logreg__penalty': ['l1', 'l2', 'elasticnet'],
    'logreg__C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    'logreg__solver': ['liblinear', 'saga'],
    'logreg__l1_ratio': [0.0, 0.3, 0.5, 0.7, 1.0],
    'logreg__class_weight': [None, 'balanced'],
    'logreg__fit_intercept': [True, False]
}

# Stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Randomized Search (faster + effective)
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=60,
    cv=cv,
    scoring='accuracy',   # change to 'f1' or 'roc_auc' if needed
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Train
search.fit(X_train, y_train)

# Best model
log_model = search.best_estimator_

# Predictions
y_pred_log = log_model.predict(X_test)

# Save model
pickle.dump(log_model, open(r"D:\Emi Predict\Models-1\clas_log_1.pkl", "wb"))

# Results
print("Best Parameters:", search.best_params_)

accuracy = accuracy_score(y_test, y_pred_log)
precision = precision_score(y_test, y_pred_log)
recall = recall_score(y_test, y_pred_log)
f1 = f1_score(y_test, y_pred_log)

print("\n📊 Classification Metrics")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))

Fitting 5 folds for each of 60 candidates, totalling 300 fits


d:\Emi Predict\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
65 fits failed out of a total of 300.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
65 fits failed with the following error:
Traceback (most recent call last):
  File "d:\Emi Predict\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "d:\Emi Predict\.venv\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Emi Predict\.venv\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_

Best Parameters: {'poly__degree': 2, 'logreg__solver': 'saga', 'logreg__penalty': 'elasticnet', 'logreg__l1_ratio': 0.7, 'logreg__fit_intercept': False, 'logreg__class_weight': None, 'logreg__C': 100}

📊 Classification Metrics
Accuracy  : 0.8635
Precision : 0.7203
Recall    : 0.6474
F1 Score  : 0.6819

Confusion Matrix:
[[49435  3916]
 [ 5494 10086]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.93      0.91     53351
           1       0.72      0.65      0.68     15580

    accuracy                           0.86     68931
   macro avg       0.81      0.79      0.80     68931
weighted avg       0.86      0.86      0.86     68931



RANDOM FOREST CLASSIFICATION

In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = model.predict(X_test)


pickle.dump(rf_model, open(r"D:\Emi Predict\Models-1\clas_RF.pkl", "wb"))

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_test, y_pred_rf)
precision = precision_score(y_test, y_pred_rf)
recall = recall_score(y_test, y_pred_rf)
f1 = f1_score(y_test, y_pred_rf)

print("📊 Classification Metrics")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

# Full Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

📊 Classification Metrics
Accuracy  : 0.8660
Precision : 0.7522
Recall    : 0.6069
F1 Score  : 0.6718

Confusion Matrix:
[[50236  3115]
 [ 6125  9455]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.94      0.92     53351
           1       0.75      0.61      0.67     15580

    accuracy                           0.87     68931
   macro avg       0.82      0.77      0.79     68931
weighted avg       0.86      0.87      0.86     68931



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
import pickle
from sklearn.metrics import roc_auc_score

# Base model
rf = RandomForestClassifier(random_state=42)

# Parameter grid
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

# Random search
rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,              # try 30 combinations
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    scoring='roc_auc'       # better for EMI prediction
)

# Fit
rf_random.fit(X_train, y_train)

# Best model
rf_model = rf_random.best_estimator_

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Save correct model ✅
pickle.dump(rf_model, open(r"D:\Emi Predict\Models-1\clas_RF.pkl", "wb"))

# Best params
print("Best Parameters:", rf_random.best_params_)

accuracy = accuracy_score(y_test, y_pred_rf)
precision = precision_score(y_test, y_pred_rf)
recall = recall_score(y_test, y_pred_rf)
f1 = f1_score(y_test, y_pred_rf)

print("📊 Classification Metrics")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

# Full Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))


print("ROC-AUC:", roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1]))

BEST MODEL PREDICTION

In [23]:
models = {
    "Logistic Regression": y_pred,
    "Random Forest": y_pred_rf,
    "XGBoost": y_pre_xgb,
    "Logistic Regression_1": y_pred_log
}

results = []

for model_name, preds in models.items():

    acc, prec, rec, f1 = evaluate_model(y_test, preds)

    results.append({
        "Model": model_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1
    })

results_df = pd.DataFrame(results)

print(results_df)

                   Model  Accuracy  Precision    Recall  F1 Score
0    Logistic Regression  0.857916   0.851584  0.857916  0.853059
1          Random Forest  0.865953   0.859877  0.865953  0.860625
2                XGBoost  0.872989   0.870678  0.872989  0.871645
3  Logistic Regression_1  0.863487   0.859374  0.863487  0.860840


In [18]:
def evaluate_model(y_test, y_pred):

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    return accuracy, precision, recall, f1

In [24]:
best_model = results_df.sort_values(by='Accuracy', ascending=False)

print("Best Performing Model:\n")
print(best_model.head(1))

Best Performing Model:

     Model  Accuracy  Precision    Recall  F1 Score
2  XGBoost  0.872989   0.870678  0.872989  0.871645
